# Cohort Retention Analysis
**RetailPulse | Business Analytics**

Monthly cohort analysis tracking what percentage of customers return to purchase in subsequent months.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

conn = psycopg2.connect(
    host='postgres', dbname='retailpulse',
    user='retailpulse', password='retailpulse123'
)
print('Connected to PostgreSQL')

In [ ]:
# Load cohort retention matrix from Gold layer
df = pd.read_sql("""
    SELECT cohort_month, period_number, active_users, cohort_size, retention_rate
    FROM gold.agg_cohort_retention
    ORDER BY cohort_month, period_number
""", conn)

df['cohort_month'] = pd.to_datetime(df['cohort_month']).dt.strftime('%Y-%m')
print(f'Cohorts: {df["cohort_month"].nunique()}')
df.head()

In [ ]:
# Pivot to cohort retention matrix
retention_matrix = df.pivot(
    index='cohort_month',
    columns='period_number',
    values='retention_rate'
)
retention_matrix.columns = [f'Month {int(c)}' for c in retention_matrix.columns]

plt.figure(figsize=(14, 7))
sns.heatmap(
    retention_matrix,
    annot=True, fmt='.1f',
    cmap='YlOrRd', vmin=0, vmax=100,
    linewidths=0.5, cbar_kws={'label': 'Retention Rate (%)'}
)
plt.title('Monthly Cohort Retention Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Acquisition Cohort')
plt.tight_layout()
plt.savefig('cohort_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Average retention by period across all cohorts
avg_retention = retention_matrix.mean()

plt.figure(figsize=(10, 5))
plt.plot(avg_retention.index, avg_retention.values, marker='o', linewidth=2, color='steelblue')
plt.fill_between(range(len(avg_retention)), avg_retention.values, alpha=0.2, color='steelblue')
plt.title('Average Customer Retention Curve', fontsize=14, fontweight='bold')
plt.xlabel('Months After First Purchase')
plt.ylabel('Average Retention Rate (%)')
plt.xticks(range(len(avg_retention)), avg_retention.index, rotation=45)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Month-1 retention: {avg_retention.iloc[1]:.1f}%')
print(f'Month-3 retention: {avg_retention.iloc[3]:.1f}%')

## Key Takeaways
- Month 0 = 100% (acquisition month)
- Steep drop in Month 1 is normal in e-commerce (30-40% typical)
- Cohorts with higher Month-3 retention have better LTV
- Target 'At Risk' RFM segment to improve Month-1 retention